# Assignment 2: The Dirty Ledger - Cleaning & Audit Work
**Course:** ICA-F26 The Prices Desk  
**Student Work:** Loading, cleaning, and data quality audit of `teaching_prices_messy.csv`.

## 1. Setup and Data Loading
First, we import required libraries and load the raw messy dataset.

In [1]:
import pandas as pd
import numpy as np

# Load raw data
df_raw = pd.read_csv('teaching_prices_messy.csv')
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head(10)

Raw dataset shape: (97, 6)


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
5,2026-01-01,Pulse C,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
6,2026-01-01,Sweetener D,City 1,149.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
7,2026-01-01,Sweetener D,City 2,155.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
8,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
9,2026-01-08,Staple A,City 2,105.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 2. Exploratory Data Analysis & Problem Diagnosis
We inspect data types, missing values, duplicates, and anomalous price entries.

In [2]:
# Check missing values
print("Missing values per column:\n", df_raw.isnull().sum())

# Check duplicates
print(f"Number of duplicate rows: {df_raw.duplicated().sum()}")
display(df_raw[df_raw.duplicated(keep=False)])

# Check unique items and formatting issues
print("Unique items:", df_raw['item'].unique())
print("Non-numeric price entries inspection:")
display(df_raw[df_raw['price'].astype(str).str.contains(',', na=False) | (df_raw['price'] == '9999') | df_raw['price'].isna()])

Missing values per column:
 date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64
Number of duplicate rows: 1


,date,item,city,price,unit,data_label
8,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
96,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


Unique items: ['Staple A' 'Oil B' 'Pulse C' 'Sweetener D' 'oil b']
Non-numeric price entries inspection:


,date,item,city,price,unit,data_label
5,2026-01-01,Pulse C,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
17,2026-01-15,Staple A,City 2,"1,250",PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
30,2026-01-22,Sweetener D,City 1,9999,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


### Summary of Diagnosed Issues:
1. **Duplicate Row:** Row 96 is an exact duplicate of Row 8 (`2026-01-08`, `Staple A`, `City 1`, `99.5`). *Decision: Exclude duplicate.*
2. **Missing Price (`NaN`):** Row 5 (`Pulse C`, `City 2`, `2026-01-01`). *Decision: Impute/Interpolate based on the consistent City 1 vs City 2 price delta (+6.0).* flavoring City 2 as City 1 + 6.0 = `206.0`.
3. **String Typo with Comma:** Row 17 (`Staple A`, `City 2`, `2026-01-15`) has price `'1,250'`. *Decision: Correct typo from `1,250` to `109.0` (following trend: City 1 is 103.0, expected City 2 is 109.0).* 
4. **Outlier Placeholder (`9999`):** Row 30 (`Sweetener D`, `City 1`, `2026-01-22`) has price `9999`. *Decision: Correct to expected linear trend value (~`158.0`) or interpolate.* 
5. **Mislabeled Item Case & Value:** Row 40 has item `'oil b'` with price `108.5` on `2026-02-05`, while `Staple A` for City 1 is missing on that date. *Decision: Correct item name from `'oil b'` to `'Staple A'`.*

## 3. Cleaning Code & Court Verdicts
We test and refine our cleaning steps line by line.

In [3]:
# Create working copy
df_clean = df_raw.copy()

# 1. Drop duplicate rows
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# 2. Standardize item names (fix casing 'oil b' -> 'Oil B')
df_clean['item'] = df_clean['item'].replace({'oil b': 'Oil B'})

# 3. Clean price column: remove commas, convert to numeric
df_clean['price'] = df_clean['price'].astype(str).str.replace(',', '', regex=True)
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')

# 4. Address specific anomalies & placeholders (9999, 1250 typo, missing values)
# Fix row 17 (Staple A City 2 on 2026-01-15 recorded as 1250 -> corrected to 109.0)
df_clean.loc[(df_clean['date'] == '2026-01-15') & (df_clean['item'] == 'Staple A') & (df_clean['city'] == 'City 2'), 'price'] = 109.0

# Fix row 30 (Sweetener D City 1 on 2026-01-22 recorded as 9999 -> corrected via interpolation/trend to 157.5)
df_clean.loc[(df_clean['date'] == '2026-01-22') & (df_clean['item'] == 'Sweetener D') & (df_clean['city'] == 'City 1'), 'price'] = 157.5

# Fix row 40 (Mislabeled 'oil b' with price 108.5 -> item is Staple City 1)
df_clean.loc[(df_clean['date'] == '2026-02-05') & (df_clean['city'] == 'City 1') & (df_clean['price'] == 108.5), 'item'] = 'Staple A'

# Fix missing Pulse C City 2 on 2026-01-01 (City 1 is 200.0, City 2 is 206.0)
df_clean.loc[(df_clean['date'] == '2026-01-01') & (df_clean['item'] == 'Pulse C') & (df_clean['city'] == 'City 2'), 'price'] = 206.0

print(f"Cleaned dataset shape: {df_clean.shape}")
print(df_clean.isnull().sum())

Cleaned dataset shape: (96, 6)
date          0
item          0
city          0
price         0
unit          0
data_label    0
dtype: int64


## 4. Before and After Comparison

In [4]:
print("--- BEFORE CLEANING SUMMARY ---")
display(df_raw.describe(include='all'))

print("--- AFTER CLEANING SUMMARY ---")
display(df_clean.describe(include='all'))

--- BEFORE CLEANING SUMMARY ---


,date,item,city,price,unit,data_label
count,97,97,97,96,97,97
unique,12,5,2,70,1,1
top,2026-01-08,Staple A,City 1,202.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
freq,9,24,49,5,97,97


--- AFTER CLEANING SUMMARY ---


,date,item,city,price,unit,data_label
count,96,96,96,96.000000,96,96
unique,12,4,2,NaN,1,1
top,2026-01-01,Staple A,City 1,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
freq,8,24,48,NaN,96,96
mean,NaN,NaN,NaN,202.166667,NaN,NaN
std,NaN,NaN,NaN,79.374949,NaN,NaN
min,NaN,NaN,NaN,99.500000,NaN,NaN
25%,NaN,NaN,NaN,142.250000,NaN,NaN
50%,NaN,NaN,NaN,192.250000,NaN,NaN
75%,NaN,NaN,NaN,234.625000,NaN,NaN


## 5. Verification & Final Inspection
Checking observation counts per item and city to ensure structural integrity.

In [5]:
print(df_clean.groupby(['item', 'city']).size())

item         city  
Oil B        City 1    12
             City 2    12
Pulse C      City 1    12
             City 2    12
Staple A     City 1    12
             City 2    12
Sweetener D  City 1    12
             City 2    12
dtype: int64
